
# Silver Ubicaciones

**Fuente:** `brz_dataco.sharepoint.raw_supply_chain` (histórico completo acumulado)

**Destino:** Tabla de **Ubicaciones**, en `slv_dataco.supply_chain`

**Grano:** una combinación única de `market` + `order_region` + `order_country` + `order_state` + `order_city` + `order_zipcode`

## Qué hace

1. Lee todo el histórico de Bronze acumulado, no solo el último corte.
2. Deduplica por la clave compuesta de 6 columnas.
3. Prioriza el registro correspondiente al corte más reciente mediante `dedup_reciente`, ordenando por `fecha_fin_corte` descendente.
4. Limpia y normaliza los datos:

   * Campos de texto: `trim` + mayúsculas.
   * `latitude` / `longitude`: `decimal(9,6)`.
5. Conserva las columnas de trazabilidad provenientes de Bronze.
6. Escribe la tabla Silver mediante `overwrite` completo.

## Trazabilidad conservada

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`

## Resultado esperado

134 combinaciones únicas de ubicación sobre 1194 filas de Bronze para el histórico actualmente cargado.

## Nota de diseño

A diferencia de las tablas de **Clientes** y **Productos**, esta tabla utiliza `dedup_reciente` para priorizar la versión correspondiente al corte más reciente cuando existen múltiples registros para una misma combinación de ubicación.

Esto permite mantener un resultado reproducible y alineado con la lógica definida para esta dimensión.


In [0]:
# ============================================================
# Celda 1: Parámetros + lectura de Bronze + helpers
# ------------------------------------------------------------
# Mismo patrón que silver_clientes/silver_productos. Junto los
# widgets, la lectura de Bronze y los helpers en una sola celda
# para evitar el problema de estado perdido que tuvimos antes.
# ============================================================

dbutils.widgets.text("bronze_catalog", "brz_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "sharepoint", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_supply_chain", "Tabla Bronze")

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

BRONZE_TABLE_FQN = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"
SILVER_UBICACIONES_FQN = f"{silver_catalog}.{silver_schema}.md_ubicaciones"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)

df_bronze = spark.table(BRONZE_TABLE_FQN)

from pyspark.sql import functions as F, Window

def clean_str(col_name: str, default: str = "") -> F.Column:
    return F.upper(F.trim(F.coalesce(F.col(col_name).cast("string"), F.lit(default))))

def clean_num(col_name: str, cast_type, default: str = "0") -> F.Column:
    trimmed = F.trim(F.col(col_name).cast("string"))
    nulled = F.when((trimmed.isNull()) | (trimmed == ""), None).otherwise(trimmed)
    return F.coalesce(nulled, F.lit(default)).cast(cast_type)

def dedup(df, partition_cols: list):
    w = Window.partitionBy(*partition_cols).orderBy(F.monotonically_increasing_id())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

print(f"Total filas en Bronze: {df_bronze.count()}")
print("✓ Helpers definidos: clean_str, clean_num, dedup")

In [0]:
# ============================================================
# INICIO DE LOG DE EJECUCIÓN — slv_locations
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "slv_locations"
CAPA = "silver"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
[c for c in df_bronze.columns if any(k in c.lower() for k in ["market", "region", "order_country", "order_state", "order_city", "order_zipcode", "latitude", "longitude"])]

In [0]:
# ============================================================
# Celda 2: Deduplicación DETERMINÍSTICA + construcción de md_ubicaciones
# ------------------------------------------------------------
# Cambio vs. silver_clientes/silver_productos: en vez de ordenar
# por monotonically_increasing_id() (arbitrario), ordenamos por
# <COLUMNA_TIMESTAMP> descendente -> ante un duplicado, siempre
# nos quedamos con el registro cargado más recientemente.
# Reproducible: correr esto 2 veces da el mismo resultado.
#
# Se conservan las columnas de trazabilidad provenientes de
# Bronze para mantener la trazabilidad de extremo a extremo.
# ============================================================

def dedup_reciente(df, partition_cols: list, orden_col: str):
    w = Window.partitionBy(*partition_cols).orderBy(F.col(orden_col).desc())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

df_ubicaciones_dedup = dedup_reciente(
    df_bronze,
    ["market", "order_region", "order_country", "order_state", "order_city", "order_zipcode"],
    orden_col="fecha_fin_corte"
)

df_silver_ubicaciones = df_ubicaciones_dedup.select(
    clean_str("market").alias("market"),
    clean_str("order_region").alias("order_region"),
    clean_str("order_country").alias("order_country"),
    clean_str("order_state").alias("order_state"),
    clean_str("order_city").alias("order_city"),
    clean_str("order_zipcode").alias("order_zipcode"),
    clean_num("latitude", "decimal(9,6)").alias("latitude"),
    clean_num("longitude", "decimal(9,6)").alias("longitude"),

    # --------------------------------------------------------
    # Trazabilidad
    # --------------------------------------------------------
    F.col("fecha_inicio_corte"),
    F.col("fecha_fin_corte"),
    F.col("fecha_carga"),
    F.col("id_ejecucion")
)

print(f"Filas antes del dedup   : {df_bronze.count()}")
print(f"Filas después del dedup : {df_silver_ubicaciones.count()}")

display(df_silver_ubicaciones.limit(10))

In [0]:
# ============================================================
# Celda 3: Escritura a silver_ubicaciones
# ------------------------------------------------------------
# overwrite completo, mismo patrón que silver_clientes y
# silver_productos.
# ============================================================

(
    df_silver_ubicaciones.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_UBICACIONES_FQN)
)

print(f"✓ {df_silver_ubicaciones.count()} filas escritas en {SILVER_UBICACIONES_FQN}")

In [0]:
%sql

SELECT *
FROM slv_dataco.supply_chain.md_ubicaciones
LIMIT 10;

In [0]:
TABLA_DESTINO = "slv_dataco.supply_chain.md_ubicaciones"   # cambia por notebook (ver tabla)
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")